# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbrahimAmr-PR/flyrank-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = One unique pseudonymized content item (page).
Time Window: Mid-panel observation window (month=2026-03) evaluated for feature extraction, with the final month (June 2026) reserved as a sealed test holdout set

In [ ]:
from pathlib import Path
import pandas as pd

path = '/content/content_refresh_anonymized (1).csv'
df = pd.read_csv(path)

total_rows = len(df)
unique_content = df['content_id'].nunique()

print(f"Total rows: {total_rows}")
print(f"Unique content IDs: {unique_content}")
print(f"Grain Check Passed: {total_rows == unique_content}")

Total rows: 30000
Unique content IDs: 30000
Grain Check Passed: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Feature Bucket: search_volume, competition, cpc, content_type, main_intent.
2. Label Bucket: is_declining (Binary proxy: 1 if trend_direction == 'down', else 0).
3. Context Bucket: content_id, last_updated.
4. Excluded Bucket: Post-decision performance metrics and direct trend text strings (trend_direction).

Why Excluded? Including raw trend text or future performance logs causes direct target leakage, giving the model hindsight bias that wouldn't exist at decision time.

In [ ]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

features = ['search_volume', 'competition', 'cpc', 'content_type', 'main_intent']
label = ['is_declining']
context = ['content_id']
excluded = ['trend_direction']

print("Features Bucket:", features)
print("Label Bucket:", label)
print("Context Bucket:", context)
print("Excluded Bucket:", excluded)

Features Bucket: ['search_volume', 'competition', 'cpc', 'content_type', 'main_intent']
Label Bucket: ['is_declining']
Context Bucket: ['content_id']
Excluded Bucket: ['trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Contract Verification Queries:

Query 1: Verify 1:1 grain mapping.

Query 2: Check overall slice row counts and label distribution.

Query 3: Missing value check across critical feature columns.

In [ ]:
assert len(df) == df['content_id'].nunique(), "Grain Violation!"

print("--- Data Slice Counts ---")
print(f"Total Rows: {len(df)}")
print(f"Target Distribution:\n{df['is_declining'].value_counts(normalize=True)}")

print("\n--- Missing Value Check ---")
print(df[features + label].isna().sum())

--- Data Slice Counts ---
Total Rows: 30000
Target Distribution:
is_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64

--- Missing Value Check ---
search_volume    2468
competition      2468
cpc              2468
content_type        0
main_intent      2374
is_declining        0
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitations of this Data Slice:

Observational Snapshots: The dataset provides static historical performance features but lacks logged editorial intervention timestamps (e.g., exact dates past content updates occurred).

No Causal Lift Direct Measurement: The data identifies pages experiencing visibility decay, but cannot directly measure the true causal traffic lift of a hypothetical refresh action without controlled A/B experiment logs.

In [ ]:
print("Feature Boundary Limits (Summary Statistics):")
print(df[['search_volume', 'competition', 'cpc']].describe())

Feature Boundary Limits (Summary Statistics):
       search_volume   competition           cpc
count   27532.000000  27532.000000  27532.000000
mean      158.882391      0.146954      0.485342
std      1518.270825      0.285241      2.101560
min         0.000000      0.000000      0.000000
25%         0.000000      0.000000      0.000000
50%        10.000000      0.000000      0.000000
75%        20.000000      0.130000      0.000000
max     74000.000000      1.000000    100.360000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.